In [13]:
from google.cloud import bigquery
from google.cloud import storage
import requests
import math
import os
import pandas as pd
import io
import google.cloud.exceptions
import logging
import csv
from concurrent.futures import ThreadPoolExecutor

In [14]:
def return_extracted_vins():
    client = bigquery.Client(project="dt-maxa-sandbox-dev")
    query = """
                SELECT distinct VIN FROM `dt-maxa-sandbox-dev.uncleaned_data.raw_car_data`
                WHERE cylinders is null or 
                condition is null or 
                drive is null or 
                fuel is null or
                manufacturer is null or 
                model is null or
                size is null or 
                type is null and
                VIN is not null
            """
    query_job = client.query(query)
    results = query_job.result().to_dataframe()
    csv_string = results.to_csv(index=False)
    vin_list = csv_string.split('\n')[2:]
    return vin_list


In [12]:
def write_updated_values_to_csv(total_tasks, task_id):
    storage_client = storage.Client()
    bucket = storage_client.bucket('landing-zone-used-car-data')
    blob = bucket.blob('used-card-data-enriched.csv')

    print(task_id)
    vin_list = return_extracted_vins()

    # curr_job_vin_batch_size = math.ceil(len(vin_list)/total_tasks)

    # start = (task_id)*curr_job_vin_batch_size
    # end = min(len(vin_list),start+curr_job_vin_batch_size) 

    # curr_job_vin_list = vin_list[start:end]

    # print(f"Length of vin list is {len(vin_list)}")
    # print(f"I'm processing from {start}-{end}")
    # print(f"I'm processing {len(curr_job_vin_list)} VINs")

    # curr_job_vin_list = vin_list[task_id*curr_job_vin_batch_size:(task_id*curr_job_vin_batch_size)+(curr_job_vin_batch_size-1)]

    while len(curr_job_vin_list) != 0:
        batch_vin_list = curr_job_vin_list[0:min(50,len(curr_job_vin_list))]

        updated_values = return_updated_values(batch_vin_list)

        curr_job_vin_list = curr_job_vin_list[min(50,len(curr_job_vin_list)):]

        print(f"curr_job_vin_list size {len(curr_job_vin_list)}")


        with ThreadPoolExecutor(max_workers=30) as executor:
            updated_values_df = pd.DataFrame(updated_values)
            updated_values_csv = updated_values_df.to_csv()


'4x2'